<a href="https://colab.research.google.com/github/AetherLewis/Jarvis/blob/main/ollama_colab_free_server_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

An LLM server that runs Ollama on Google Colab's GPU and exposes the Ollama endpoint via an ngrok tunnel.

Key Features
- ✅ Completely free. No data sent to external APIs — privacy protected via local inference
- 🎛️ Select a model via UI; pull and launch are fully automated
- 🚀 Instantly connectable from Continue or Claude Code


> ※ Append `/v1` to the base URL for Codex and other OpenAI-compatible clients. Example: `https://xxxx.ngrok-free.app/v1`

In [1]:
#@title 📋 Model Registry

# @markdown ### Model Configuration
# @markdown > Enter the model names you want to add, separated by commas. Guide: `8B` recommended · `14B` feasible · `20B+` slow
# @markdown > - Find official model names here: https://ollama.com/search

model_list = "\"qwen3:8b, qwen2.5-coder:latest, qwen2.5-coder:3b\"" #@param {type:"string"}

# @markdown ### context length (num_ctx)
# @markdown > T4 (16GB VRAM) Guide
# @markdown > | Model Size | Rec. ctx | Notes |
# @markdown > |---|---|---|
# @markdown > | ~4B  | 32768 | Plentiful |
# @markdown > | ~8B  | 16384 | Standard |
# @markdown > | ~14B | 8192  | VRAM tight |
# @markdown > | ~20B+ | 4096 | Min. usage |
# @markdown >
# @markdown > Default (0) lets Ollama decide (equivalent to 4096).
num_ctx = 16384 #@param {type:"integer"}

AVAILABLE_MODELS = [
    model.strip()
    for model in model_list.split(',')
    if model.strip()
]

if not AVAILABLE_MODELS:
    raise ValueError("❌ Model list is empty. Please enter at least one model.")

import ipywidgets as widgets
from IPython.display import display

header = widgets.HTML(
    '<h3>📦 Select a Model</h3>'
    '<p style="margin: 5px 0 10px 0; font-size: 13px;">'
    'Select one model to launch, then run the next cell.</p>'
)

model_selector = widgets.RadioButtons(
    options=AVAILABLE_MODELS,
    value=AVAILABLE_MODELS[0],
    layout=widgets.Layout(padding='0 0 0 20px')
)

display(widgets.VBox([header, model_selector]))

print(f"✅ Model list loaded: {len(AVAILABLE_MODELS)} model(s)")
print("➡️ After selecting a model, run the next cell (Server).")


✅ Model list loaded: 3 model(s)
➡️ After selecting a model, run the next cell (Server).


In [ ]:
#@title 🚀 Ollama Colab Free Server

# @markdown > Create a free Ngrok account at the URL below and paste your auth token here.
# @markdown > - https://dashboard.ngrok.com/get-started/your-authtoken

ngrok_token = "3DiObuZdlrHRlUNw10KKUwARjnH_4C9WWGJnVhJHiGmpbCwr6" #@param {type:"string"}

MAX_HEALTH_RETRIES   = 150
HEALTH_CHECK_TIMEOUT = 2
STATUS_POLL_INTERVAL = 30

BLUE  = "\033[34m"
GREEN = "\033[32m"
GRAY  = "\033[90m"
RESET = "\033[0m"

selected_model = model_selector.value
effective_ctx  = num_ctx if num_ctx > 0 else None  # None = Ollama Default

print(f"\nOLLAMA COLAB SERVER 🚀")
print(f"{GRAY}──────────────────────────────────────────{RESET}")

print(f"  {BLUE}◦{RESET} System   Installing dependencies (zstd)")
!which zstd > /dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq zstd > /dev/null)

print(f"  {BLUE}◦{RESET} System   Installing Ollama")
!which ollama > /dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print(f"  {BLUE}◦{RESET} System   Installing pyngrok")
!python -c "import pyngrok" > /dev/null 2>&1 || pip install -q pyngrok

import re
import subprocess
import time
import os
import requests

# Validate model name to prevent shell injection
if not re.fullmatch(r'[a-zA-Z0-9._:/-]+', selected_model):
    raise ValueError(f"Invalid model name: {selected_model}")

os.environ['OLLAMA_HOST']              = '0.0.0.0:11434'
os.environ['OLLAMA_KEEP_ALIVE']        = '24h'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_FLASH_ATTENTION']   = '1'
os.environ['OLLAMA_KV_CACHE_TYPE']     = 'q8_0'

process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(MAX_HEALTH_RETRIES):
    try:
        if requests.get("http://0.0.0.0:11434/api/tags", timeout=HEALTH_CHECK_TIMEOUT).status_code == 200:
            print(f"  {GREEN}✓{RESET} Ready    Ollama server started (0.0.0.0:11434)")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(0.2)
else:
    raise RuntimeError("⚠️ Failed to confirm Ollama server startup.")

print(f"  {BLUE}◦{RESET} Network  Establishing ngrok tunnel (port: 11434)")
from pyngrok import ngrok

if not ngrok_token:
    raise ValueError("⚠️ ngrok token is not set.")

ngrok.set_auth_token(ngrok_token)
ngrok.kill()  # Release any previous session tunnels including remote endpoints
tunnel = ngrok.connect(11434)
public_url = tunnel.public_url

print(f"  {GREEN}✓{RESET} Public   {public_url}")

print(f"\n  ▶ Model    {selected_model}")
ctx_label = str(effective_ctx) if effective_ctx else "Ollama Default (4096)"
print(f"  {GRAY}  num_ctx  {ctx_label}{RESET}")
print(f"  {GRAY}└ Pulling  Download started... (first time: ~5–15 min){RESET}")
subprocess.run(
    ["/usr/local/bin/ollama", "pull", selected_model],
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    check=True
)
print(f"  {GREEN}✓{RESET} Loaded   Model loaded successfully")

# warmup: Load the model into VRAM with the specified context length
# (omitting keep_alive lets OLLAMA_KEEP_ALIVE take effect)
warmup_payload = {
    "model": selected_model,
    "prompt": "hi",
    "stream": False,
}
if effective_ctx:
    warmup_payload["options"] = {"num_ctx": effective_ctx}

try:
    requests.post(
        "http://0.0.0.0:11434/api/generate",
        json=warmup_payload,
        timeout=180
    )
    ctx_info = f" (num_ctx: {effective_ctx})" if effective_ctx else ""
    print(f"  {GREEN}✓{RESET} Warmed   Model loaded into VRAM{ctx_info}")
except requests.exceptions.RequestException as e:
    print(f"  ⚠️  Warmup   Warm-up failed: {e}")

ctx_disp = str(effective_ctx) if effective_ctx else "4096 (Ollama Default)"
print(f"\n{GRAY}──────────────────────────────────────────{RESET}")
print(f"ENDPOINT : {public_url}")
print(f"NUM_CTX  : {ctx_disp}")
print(f"{GRAY}──────────────────────────────────────────{RESET}")

print(f"Continue Extension Config (~/.continue/config.yaml)")
print(f"{GRAY}Example config for using a local LLM with the Continue extension in VS Code etc.{RESET}")
ctx_yaml = effective_ctx if effective_ctx else 4096
print(f"""
models:
  - title: {selected_model}
    provider: ollama
    model: {selected_model}
    apiBase: {public_url}
    contextLength: {ctx_yaml}
""")

print(f"Claude Code Setup (shell env)")
print(f"{GRAY}Environment variables to redirect the API endpoint to this server for use with the claude command.{RESET}")
print(f"""
export ANTHROPIC_BASE_URL={public_url}
export ANTHROPIC_API_KEY=dummy
claude --model {selected_model}
""")

print(f"Codex Setup (Codex extension for VS Code etc. / CLI)")
print(f"{GRAY}Example config for using a local LLM with the Codex extension or CLI.{RESET}")
print(f"""
{GRAY}~/.codex/config.toml{RESET}
model = \"{selected_model}\"

{GRAY}shell env{RESET}
export OPENAI_BASE_URL={public_url}/v1
export OPENAI_API_KEY=dummy
code .    {GRAY}# When launching the extension{RESET}
codex     {GRAY}# When launching the CLI{RESET}
""")

try:
    start_time = time.time()
    while True:
        elapsed_min = int((time.time() - start_time) / 60)
        print(f"\r  {GREEN}●{RESET} Running  Uptime: {elapsed_min}min | {public_url}", end="")
        time.sleep(STATUS_POLL_INTERVAL)
except KeyboardInterrupt:
    print(f"\n\n  {GRAY}Session terminated.{RESET}")
finally:
    ngrok.disconnect(public_url)



OLLAMA COLAB SERVER 🚀
──────────────────────────────────────────
  ◦ System   Installing dependencies (zstd)
  ◦ System   Installing Ollama
  ◦ System   Installing pyngrok
  ✓ Ready    Ollama server started (0.0.0.0:11434)
  ◦ Network  Establishing ngrok tunnel (port: 11434)
  ✓ Public   https://random-brethren-willfully.ngrok-free.dev

  ▶ Model    qwen2.5-coder:latest
    num_ctx  16384
  └ Pulling  Download started... (first time: ~5–15 min)
  ✓ Loaded   Model loaded successfully
  ✓ Warmed   Model loaded into VRAM (num_ctx: 16384)

──────────────────────────────────────────
ENDPOINT : https://random-brethren-willfully.ngrok-free.dev
NUM_CTX  : 16384
──────────────────────────────────────────
Continue Extension Config (~/.continue/config.yaml)
Example config for using a local LLM with the Continue extension in VS Code etc.

models:
  - title: qwen2.5-coder:latest
    provider: ollama
    model: qwen2.5-coder:latest
    apiBase: https://random-brethren-willfully.ngrok-free.dev
    c